# 03 – Bronze load

Loads the 8 raw Berka files into Delta tables under `workspace.bronze`. No type casting: all columns stay as strings, and two metadata columns are added.

## Settings and read function

One function reads the raw files and adds two metadata columns to each row: `_ingest_ts` (load time) and `_source_file` (the file the row came from).

`_source_file` comes from the hidden `_metadata` column that Databricks provides for file sources.

Output: 3 rows from `district`. `_source_file` shows the full file path and `_ingest_ts` the current time.

In [0]:
from pyspark.sql import functions as F

RAW_PATH = "/Volumes/workspace/bronze/raw/"
BRONZE_SCHEMA = "workspace.bronze"

# file name -> bronze table name ("order" is a SQL reserved word)
table_names = {
    "account": "account",
    "card": "card",
    "client": "client",
    "disp": "disp",
    "district": "district",
    "loan": "loan",
    "order": "orders",
    "trans": "trans",
}


def read_raw(file_name):
    path = RAW_PATH + file_name + ".csv"
    df_raw = spark.read.option("header", True).option("sep", ";").csv(path)
    df_with_ts = df_raw.withColumn("_ingest_ts", F.current_timestamp())
    df_with_meta = df_with_ts.withColumn("_source_file", F.col("_metadata.file_path"))
    return df_with_meta


df_test = read_raw("district")
display(df_test.select("A1", "A2", "_ingest_ts", "_source_file").limit(3))

## Write to bronze Delta tables

Reads the 8 raw files and writes them as Delta tables under `workspace.bronze`. Column names are unchanged; only `_ingest_ts` and `_source_file` are added. The `order` file is written to the `orders` table (`order` is a SQL reserved word).

Write mode is `overwrite`: re-running the notebook replaces the tables without duplicates. Delta creates a new version on every write.

Output: one line per table (`written: workspace.bronze.<table>`).

In [0]:
for file_name in table_names:
    table_name = table_names[file_name]
    full_table_name = BRONZE_SCHEMA + "." + table_name

    df_bronze = read_raw(file_name)

    (df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name))

    print("written:", full_table_name)

## Bronze row counts

Checks that the bronze row counts match the expected values (the raw file counts confirmed in 02_raw_data_check).

Output: `OK` or `MISMATCH` per table, with the total at the end. Expected total: 1,079,680.

In [0]:
expected_counts = {
    "account": 4500,
    "card": 892,
    "client": 5369,
    "disp": 5369,
    "district": 77,
    "loan": 682,
    "orders": 6471,
    "trans": 1056320,
}

total_rows = 0

for table_name in expected_counts:
    full_table_name = BRONZE_SCHEMA + "." + table_name
    actual = spark.table(full_table_name).count()
    expected = expected_counts[table_name]

    if actual == expected:
        status = "OK"
    else:
        status = "MISMATCH"

    print(status, full_table_name, actual, "/", expected)
    total_rows = total_rows + actual

print("total:", total_rows)

## DESCRIBE DETAIL and DESCRIBE HISTORY

Confirms that the bronze tables are Delta and shows the version log kept by Delta. Example table: `trans`.

`DESCRIBE DETAIL`: `format` is delta. `numFiles` and `sizeInBytes` show how many Parquet files the table has and its size on disk (expected to be smaller than the ~69 MB CSV, because Parquet is compressed).

`DESCRIBE HISTORY`: every write is a `version`. `operation` is CREATE OR REPLACE TABLE AS SELECT, and `numOutputRows` in `operationMetrics` is 1056320.

In [0]:
%sql
DESCRIBE DETAIL workspace.bronze.trans

In [0]:
%sql
DESCRIBE HISTORY workspace.bronze.trans